# **Data Collection Notebook**

## Objectives

* Fetch raw dataset from Kaggle and save to inputs/datasets/raw/

## Inputs

* Kaggle JSON file - the authentication token.
  
## Outputs

* Generate datasets:
  * inputs/datasets/raw/house-price-20211124T154130Z-001/house-price/house_prices_records.csv
  



---

# Change working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [ ]:
import os
current_dir = os.getcwd()
current_dir

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chir() defines the new current directory

In [ ]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

Confirm the new current directory

In [ ]:
current_dir = os.getcwd()
current_dir

# Import Packages

In [ ]:
%pip install -r requirements.txt

---

# Fetch data from Kaggle

Install Kaggle package to fetch data

In [ ]:
%pip install kaggle==1.5.12

NB - Ensure you have a valid kaggle.json file

In [ ]:
import os
os.environ['KAGGLE_CONFIG_DIR'] = os.getcwd()
! chmod 600 kaggle.json

We are using the following [Kaggle URL](https://www.kaggle.com/datasets/codeinstitute/housing-prices-data)

Get the dataset path from the Kaggle url
* When you are viewing the dataset at Kaggle, check what is after https://www.kaggle.com/ .

Define the Kaggle dataset, and destination folder and download it.

In [ ]:
KaggleDatasetPath = "codeinstitute/housing-prices-data/data"
DestinationFolder = "inputs/datasets/raw"   
! kaggle datasets download -d {KaggleDatasetPath} -p {DestinationFolder}

Unzip the downloaded file, delete the zip file and delete the kaggle.json file:

In [ ]:
! unzip {DestinationFolder}/*.zip -d {DestinationFolder} \
  && rm {DestinationFolder}/*.zip \
  && rm kaggle.json

---

# Load and Inspect Kaggle data

In [ ]:
import pandas as pd
df = pd.read_csv(f"inputs/datasets/raw/house-price-20211124T154130Z-001/house-price/house_prices_records.csv")
df.head()

**DataFrame Summary**

We can see that there are 1460 entries in the dataset.

We have 24 total variables; our target, Sale Price, and 23 features.

20 variables are numerical, 4 are categorical variables.

A decription for the variable names is included in the [text file](/workspaces/heritage-housing-issues-milestone/inputs/datasets/raw/house-metadata.txt).

An immediate observation is that EnclosedPorch and WoodDeckSF have a very high null count, with over 90% of data missing.

In [ ]:
df.info()

**Drop variables with high null count**

We have decided to drop EnclosedPorch and WoodDeckSF from our main dataset, for the following reasons:
* The high levels of missing data would have no meaningful insight on correlation. 
* We will return a cleaner analysis with less confusing factors. 
* Imputating such a high percentage of values would be meaningless.

In [ ]:
# Drop variables with high null count
vars_to_drop = ['EnclosedPorch', 'WoodDeckSF']
df_cleaned = df.drop(columns=vars_to_drop, axis=1)

# Check cleaned dataframe shape against original dataframe
print(f"Original DataFrame: {df.shape[1]} columns, {df.shape[0]} rows")
print(f"Cleaned DataFrame: {df_cleaned.shape[1]} columns, {df_cleaned.shape[0]} rows")
print(f"\nColumns dropped: {df.shape[1] - df_cleaned.shape[1]}")

# Print which columns were dropped
dropped_cols = set(df.columns) - set(df_cleaned.columns)
print(f"Dropped column names: {dropped_cols}")

In [ ]:
# Inspect cleaned dataset
df_cleaned.info()

In [ ]:
# Create collections folder and save cleaned dataset
import os
try:
    os.makedirs(name='outputs/datasets/collection')
except Exception as e:
    print(e)

df_cleaned.to_csv('outputs/datasets/collection/house_prices', index=False)

# Push files to Repo